# Fake News Classifier: Baseline + BERT with LIME
This notebook covers:
1. Data Loading (LIAR + FakeNewsNet)
2. TF-IDF + Logistic Regression Baseline
3. Fine-tuning `bert-base-uncased` with HuggingFace Trainer
4. Evaluation (Accuracy, F1, Confusion Matrix)
5. LIME Explainability

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn lime matplotlib seaborn accelerate torch

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset, concatenate_datasets, Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import matplotlib.pyplot as plt
import seaborn as sns
from lime.lime_text import LimeTextExplainer

## 1. Data Loading (LIAR + FakeNewsNet)
We will load LIAR using the `datasets` library. We also provide the structure to load and concatenate FakeNewsNet. For simplicity, we map labels to a binary format: 1 for Real, 0 for Fake.

In [ ]:
# Load LIAR dataset
liar = load_dataset('liar')

# LIAR labels: 0: pants-fire, 1: false, 2: barely-true, 3: half-true, 4: mostly-true, 5: true
# Map 0, 1, 2 to Fake (0); 3, 4, 5 to Real (1)
def map_liar_label(example):
    label = 0 if example['label'] in [0, 1, 2] else 1
    return {'text': example['statement'], 'label': label}

liar_train = liar['train'].map(map_liar_label, remove_columns=liar['train'].column_names)
liar_val = liar['validation'].map(map_liar_label, remove_columns=liar['validation'].column_names)
liar_test = liar['test'].map(map_liar_label, remove_columns=liar['test'].column_names)

# For FakeNewsNet, we mock the loading here. In practice, you would load it from a CSV and concatenate.
# Example:
# fnn_df = pd.read_csv('fakenewsnet.csv')
# fnn_dataset = Dataset.from_pandas(fnn_df)
# train_dataset = concatenate_datasets([liar_train, fnn_dataset])

train_dataset = liar_train
val_dataset = liar_val
test_dataset = liar_test

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

## 2. Baseline: TF-IDF + Logistic Regression

In [ ]:
# Extract text and labels
X_train, y_train = train_dataset['text'], train_dataset['label']
X_test, y_test = test_dataset['text'], test_dataset['label']

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Predict and evaluate
lr_preds = lr_model.predict(X_test_tfidf)
print("Baseline Accuracy:", accuracy_score(y_test, lr_preds))
print("Baseline F1:", f1_score(y_test, lr_preds))

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

plot_cm(y_test, lr_preds, "Baseline TF-IDF + LR Confusion Matrix")

## 3. Fine-Tuning `bert-base-uncased`

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_val = tokenized_val.remove_columns(['text'])
tokenized_test = tokenized_test.remove_columns(['text'])

tokenized_train.set_format('torch')
tokenized_val.set_format('torch')
tokenized_test.set_format('torch')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# trainer.train() # Uncomment to start training

## 4. Evaluation & Saving Model

In [ ]:
# Evaluate on test set (assuming model has been trained)
# predictions = trainer.predict(tokenized_test)
# preds = np.argmax(predictions.predictions, axis=-1)

# print("BERT Accuracy:", accuracy_score(y_test, preds))
# print("BERT F1:", f1_score(y_test, preds))
# plot_cm(y_test, preds, "BERT Confusion Matrix")

In [ ]:
# Save the model and tokenizer to disk
model_path = "./fake_news_bert_model"
# trainer.save_model(model_path)
# tokenizer.save_pretrained(model_path)
# print(f"Model saved to {model_path}")

## 5. LIME Explainability
We add LIME (Local Interpretable Model-agnostic Explanations) to interpret model predictions for specific texts. The function returns the top 5 phrases with contribution scores.

In [ ]:
class BERT_Explainer:
    def __init__(self, model_path_or_obj, tokenizer_obj=None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        if isinstance(model_path_or_obj, str):
            self.model = AutoModelForSequenceClassification.from_pretrained(model_path_or_obj).to(self.device)
            self.tokenizer = AutoTokenizer.from_pretrained(model_path_or_obj)
        else:
            self.model = model_path_or_obj.to(self.device)
            self.tokenizer = tokenizer_obj
            
        self.model.eval()
        self.explainer = LimeTextExplainer(class_names=['Fake', 'Real'])
        
    def predictor_func(self, texts):
        inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probas = F.softmax(outputs.logits, dim=1).cpu().numpy()
        return probas

    def explain_prediction(self, text, num_features=5):
        """
        Returns the top `num_features` phrases that most influenced the verdict 
        with their contribution scores.
        """
        # Generate explanation
        exp = self.explainer.explain_instance(text, self.predictor_func, num_features=num_features, num_samples=100)
        
        # Get the contributions (phrase, score) tuples
        contributions = exp.as_list()
        
        probs = self.predictor_func([text])[0]
        predicted_class = 'Real' if probs[1] > probs[0] else 'Fake'
        
        print(f"Text: {text}")
        print(f"Predicted Class: {predicted_class} (Prob: {probs[1]:.4f} for Real)")
        print("\nTop Phrases & Contributions (Positive -> Real, Negative -> Fake):")
        
        for phrase, score in contributions:
            print(f"{phrase}: {score:.4f}")
            
        # Display it visually in Jupyter
        exp.show_in_notebook(text=True)
        
        return contributions

# Example usage:
# To use with the newly trained model in memory:
# explainer = BERT_Explainer(model, tokenizer)

# Or if loading from disk later:
# explainer = BERT_Explainer("./fake_news_bert_model")

# sample_text = "The stock market crashed today because of fake reports about alien invasions."
# explanations = explainer.explain_prediction(sample_text, num_features=5)
